In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

# Detect environment
if Path("/kaggle/input").exists():
    DATA_DIR = Path("/kaggle/input/ieee-fraud-detection")
else:
    DATA_DIR = Path("data/raw")

print("Using data directory:", DATA_DIR)

for file_path in DATA_DIR.iterdir():
    print(file_path)

Using data directory: data\raw
data\raw\test_identity.csv
data\raw\test_transaction.csv
data\raw\train_identity.csv
data\raw\train_transaction.csv


In [2]:
pip install dagshub mlflow scikit-learn pandas matplotlib seaborn skops --quiet

Note: you may need to restart the kernel to use updated packages.


In [3]:
import dagshub
import mlflow
import mlflow.sklearn
import joblib

dagshub.init(repo_owner='myvari', repo_name='IEEE-CIS-Fraud-Detection', mlflow=True)

Accessing as myvari

Initialized MLflow to track repo "myvari/IEEE-CIS-Fraud-Detection"

Repository myvari/IEEE-CIS-Fraud-Detection initialized!

In [4]:
from sklearn.base import BaseEstimator, TransformerMixin

class DataCleaner(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        drop_transaction_id=True,
        missing_threshold=0.95,
        drop_constant=True,
        drop_high_cardinality_cols=["DeviceInfo"]
    ):
        self.drop_transaction_id = drop_transaction_id
        self.missing_threshold = missing_threshold
        self.drop_constant = drop_constant
        self.drop_high_cardinality_cols = drop_high_cardinality_cols

    def fit(self, X, y=None):
        X = X.copy()
        X.columns = X.columns.str.replace("id-", "id_", regex=False)

        self.columns_to_drop_ = []

        if self.drop_transaction_id and "TransactionID" in X.columns:
            self.columns_to_drop_.append("TransactionID")

        drop_high_cardinality_cols = (
            self.drop_high_cardinality_cols
            if self.drop_high_cardinality_cols is not None
            else []
        )

        # drop high cardinality columns if selected
        for col in drop_high_cardinality_cols:
            if col in X.columns:
                self.columns_to_drop_.append(col)

        # drop columns if missingness above threshold
        if self.missing_threshold is not None:
            missing_rates = X.isna().mean()
            high_missing_cols = missing_rates[missing_rates > self.missing_threshold].index.tolist()
            self.columns_to_drop_.extend(high_missing_cols)

        # drop constant columns
        if self.drop_constant:
            constant_cols = [
                col for col in X.columns
                if X[col].nunique(dropna=False) <= 1
            ]
            self.columns_to_drop_.extend(constant_cols)

        self.columns_to_drop_ = sorted(set(self.columns_to_drop_))

        return self


    def transform(self, X):
        X = X.copy()
        X.columns = X.columns.str.replace("id-", "id_", regex=False)

        X = X.replace([np.inf, -np.inf], np.nan) # handle infinite values if any

        X = X.drop(columns=self.columns_to_drop_, errors="ignore")

        return X

    def set_output(self, transform="pandas"):
        self._transform_output = transform
        return self

In [5]:

class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        add_identity_feature=True,
        add_amount_features=True,
        add_time_features=True,
        add_email_groups=True,
    ):
        self.add_identity_feature = add_identity_feature
        self.add_amount_features = add_amount_features
        self.add_time_features = add_time_features
        self.add_email_groups = add_email_groups

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X.columns = X.columns.str.replace("id-", "id_", regex=False)

        if self.add_identity_feature:
            if "id_01" in X.columns: # defensive check since some versions of the dataset have inconsistent id column naming
                X["has_identity"] = X["id_01"].notna().astype(int)
            else:
                X["has_identity"] = 0

        # transaction amount features
        if self.add_amount_features and "TransactionAmt" in X.columns:
            X["TransactionAmt_decimal"] = (X["TransactionAmt"] - np.floor(X["TransactionAmt"]))

        if self.add_time_features and "TransactionDT" in X.columns:
            seconds_in_hour = 3600
            seconds_in_day = seconds_in_hour * 24

            transaction_hour = (X["TransactionDT"] // seconds_in_hour) % 24
            transaction_day = X["TransactionDT"] // seconds_in_day
            transaction_week = transaction_day // 7

            X["transaction_day"] = transaction_day
            X["transaction_week"] = transaction_week

            # cyclic encoding for time features (better reflecting of their nature)
            day_of_month = transaction_day % 31
            X["transaction_day_of_month_sin"] = np.sin(2 * np.pi * day_of_month / 31)
            X["transaction_day_of_month_cos"] = np.cos(2 * np.pi * day_of_month / 31)

            weekday = transaction_day % 7
            X["transaction_weekday_sin"] = np.sin(2 * np.pi * weekday / 7)
            X["transaction_weekday_cos"] = np.cos(2 * np.pi * weekday / 7)

            X["transaction_hour_sin"] = np.sin(2 * np.pi * transaction_hour / 24)
            X["transaction_hour_cos"] = np.cos(2 * np.pi * transaction_hour / 24)


        if self.add_email_groups:
            for col in ["P_emaildomain", "R_emaildomain"]:
                if col in X.columns:
                    X[f"{col}_group"] = X[col].apply(self._group_email_domain)

        X = X.replace([np.inf, -np.inf], np.nan)

        return X
    
    def _group_email_domain(self, value):
        if pd.isna(value):
            return "missing"

        value = str(value).lower()

        if "gmail" in value:
            return "gmail"
        elif "yahoo" in value:
            return "yahoo"
        elif value in ["hotmail.com", "outlook.com", "live.com", "msn.com"]:
            return "microsoft"
        elif "icloud" in value or "me.com" in value or "mac.com" in value:
            return "apple"
        elif "anonymous" in value:
            return "anonymous"
        elif "aol" in value:
            return "aol"
        else:
            return "other"
    
    def set_output(self, transform="pandas"):
        self._transform_output = transform
        return self

In [6]:
class NearZeroVarianceReducer(BaseEstimator, TransformerMixin):
    def __init__(self, min_unique=2, max_dominant_ratio=0.995, verbose=True):
        self.min_unique = min_unique
        self.max_dominant_ratio = max_dominant_ratio
        self.verbose = verbose

    def fit(self, X, y=None):
        X_df = X.copy()
        self.feature_names_ = X_df.columns.tolist()
        self.removed_features_ = []

        for col in self.feature_names_:
            value_counts = X_df[col].value_counts(dropna=False, normalize=True)

            if len(value_counts) < self.min_unique:
                self.removed_features_.append(col)
            elif value_counts.iloc[0] >= self.max_dominant_ratio:
                self.removed_features_.append(col)

        self.kept_features_ = [
            col for col in self.feature_names_
            if col not in self.removed_features_
        ]

        if self.verbose:
            print(f"[Near-zero variance] Removed {len(self.removed_features_)} features")
            if len(self.removed_features_) <= 50:
                print(self.removed_features_)
            else:
                print(self.removed_features_[:50], "...")

        return self

    def transform(self, X):
        X_df = X.copy()
        return X_df[self.kept_features_]

    def set_output(self, transform="pandas"):
        self._transform_output = transform
        return self

In [7]:
class XGBoostFeatureEngineerV2(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        add_identity_feature=True,
        add_amount_features=True,
        add_time_features=True,
        add_email_groups=True,
        add_uid_features=True,
        add_frequency_features=True,
        add_amount_group_features=True,
        add_d_time_features=True
    ):
        self.add_identity_feature = add_identity_feature
        self.add_amount_features = add_amount_features
        self.add_time_features = add_time_features
        self.add_email_groups = add_email_groups
        self.add_uid_features = add_uid_features
        self.add_frequency_features = add_frequency_features
        self.add_amount_group_features = add_amount_group_features
        self.add_d_time_features = add_d_time_features

    def fit(self, X, y=None):
        X = X.copy()
        X.columns = X.columns.str.replace("id-", "id_", regex=False)

        # uid columns during fit so group stats/frequencies are learned only from training data
        X_fe = self._basic_transform(X)

        self.frequency_maps_ = {}
        self.amount_group_stats_ = {}

        if self.add_frequency_features: # for rarity
            freq_cols = [
                "card1", "card2", "card3", "card5",
                "addr1", "addr2",
                "P_emaildomain", "R_emaildomain",
                "ProductCD",
                "uid_card_addr",
                "uid_card_addr_product",
                "uid_card_email"
            ]

            for col in freq_cols:
                if col in X_fe.columns:
                    self.frequency_maps_[col] = X_fe[col].value_counts(dropna=False)

        if self.add_amount_group_features and "TransactionAmt" in X_fe.columns:
            group_cols = [
                "card1",
                "addr1",
                "ProductCD",
                "uid_card_addr",
                "uid_card_addr_product",
                "uid_card_email"
            ]

            for col in group_cols:
                if col in X_fe.columns:
                    stats = X_fe.groupby(col, dropna=False)["TransactionAmt"].agg(["mean", "std", "median"])
                    self.amount_group_stats_[col] = stats

        return self

    def transform(self, X):
        X = X.copy()
        X.columns = X.columns.str.replace("id-", "id_", regex=False)

        X = self._basic_transform(X)

        if self.add_frequency_features:
            for col, freq_map in self.frequency_maps_.items():
                if col in X.columns:
                    X[f"{col}_freq"] = X[col].map(freq_map).fillna(0)

        if self.add_amount_group_features and "TransactionAmt" in X.columns:
            for col, stats in self.amount_group_stats_.items():
                if col in X.columns:
                    mean_map = stats["mean"]
                    std_map = stats["std"].replace(0, np.nan)
                    median_map = stats["median"]

                    group_mean = X[col].map(mean_map)
                    group_std = X[col].map(std_map)
                    group_median = X[col].map(median_map)

                    X[f"TransactionAmt_to_{col}_mean"] = X["TransactionAmt"] / group_mean
                    X[f"TransactionAmt_to_{col}_median"] = X["TransactionAmt"] / group_median
                    X[f"TransactionAmt_zscore_{col}"] = (X["TransactionAmt"] - group_mean) / group_std

                    X[f"TransactionAmt_to_{col}_mean"] = X[f"TransactionAmt_to_{col}_mean"].replace([np.inf, -np.inf], np.nan)
                    X[f"TransactionAmt_to_{col}_median"] = X[f"TransactionAmt_to_{col}_median"].replace([np.inf, -np.inf], np.nan)
                    X[f"TransactionAmt_zscore_{col}"] = X[f"TransactionAmt_zscore_{col}"].replace([np.inf, -np.inf], np.nan)

        return X

    def _basic_transform(self, X):
        X = X.copy()

        if self.add_identity_feature:
            identity_cols = [col for col in X.columns if col.startswith("id_")]
            if len(identity_cols) > 0:
                X["has_identity"] = X[identity_cols].notna().any(axis=1).astype(int)

        if self.add_amount_features and "TransactionAmt" in X.columns:
            X["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])
            X["TransactionAmt_decimal"] = X["TransactionAmt"] - np.floor(X["TransactionAmt"])

        if self.add_time_features and "TransactionDT" in X.columns:
            seconds_in_hour = 3600
            seconds_in_day = seconds_in_hour * 24

            transaction_hour = (X["TransactionDT"] // seconds_in_hour) % 24
            transaction_day = X["TransactionDT"] // seconds_in_day

            X["transaction_day"] = transaction_day
            X["transaction_week"] = transaction_day // 7

            day_of_month = transaction_day % 31
            X["transaction_day_of_month_sin"] = np.sin(2 * np.pi * day_of_month / 31)
            X["transaction_day_of_month_cos"] = np.cos(2 * np.pi * day_of_month / 31)

            weekday = transaction_day % 7
            X["transaction_weekday_sin"] = np.sin(2 * np.pi * weekday / 7)
            X["transaction_weekday_cos"] = np.cos(2 * np.pi * weekday / 7)

            X["transaction_hour_sin"] = np.sin(2 * np.pi * transaction_hour / 24)
            X["transaction_hour_cos"] = np.cos(2 * np.pi * transaction_hour / 24)

        if self.add_email_groups:
            for email_col in ["P_emaildomain", "R_emaildomain"]:
                if email_col in X.columns:
                    X[f"{email_col}_provider"] = X[email_col].astype(str).str.split(".").str[0]
                    X[f"{email_col}_suffix"] = X[email_col].astype(str).str.split(".").str[-1]

        if self.add_uid_features:
            X = self._add_uid_features(X)

        # trying to extract extra signals from timedelta fetaures
        if self.add_d_time_features and "transaction_day" in X.columns:
            for d_col in ["D1", "D2", "D3", "D4", "D5", "D10", "D11", "D15"]: # others have high miss
                if d_col in X.columns:
                    X[f"transaction_day_minus_{d_col}"] = X["transaction_day"] - X[d_col]

        return X

    def _add_uid_features(self, X):
        X = X.copy()

        def safe_str(col):
            if col in X.columns:
                return X[col].astype("object").where(X[col].notna(), "missing").astype(str)
            return pd.Series("missing", index=X.index)

        card1 = safe_str("card1")
        addr1 = safe_str("addr1")
        product = safe_str("ProductCD")
        p_email = safe_str("P_emaildomain")

        # attempt to capture single user characteristic
        X["uid_card_addr"] = card1 + "_" + addr1
        X["uid_card_addr_product"] = card1 + "_" + addr1 + "_" + product
        X["uid_card_email"] = card1 + "_" + p_email

        return X

    def set_output(self, transform="pandas"):
        self._transform_output = transform
        return self

In [9]:
final_run_id = "ed42f9688cfb4982b34a0f7cbf047e7f"

pipeline_path = mlflow.artifacts.download_artifacts(
    run_id=final_run_id,
    artifact_path="xgboost_fe_v2_final_full_pipeline.joblib"
)

final_pipeline = joblib.load(pipeline_path)

print("Final pipeline loaded!!")
print(type(final_pipeline))
print(final_pipeline)

Final pipeline loaded!!
<class 'sklearn.pipeline.Pipeline'>
Pipeline(steps=[('preprocessing',
                 Pipeline(steps=[('cleaner', DataCleaner()),
                                 ('feature_engineer',
                                  XGBoostFeatureEngineerV2()),
                                 ('near_zero_variance',
                                  NearZeroVarianceReducer()),
                                 ('preprocessor',
                                  ColumnTransformer(transformers=[('num',
                                                                   Pipeline(steps=[('imputer',
                                                                                    SimpleImputer(add_indicator=True,
                                                                                                  strategy='median'))]),
                                                                   <sklearn.compose._column_transformer.make...
                               feature_t

In [10]:
test_transaction = pd.read_csv(DATA_DIR / "test_transaction.csv")
test_identity = pd.read_csv(DATA_DIR / "test_identity.csv")

test_identity.columns = test_identity.columns.str.replace("id-", "id_", regex=False)

test_df = pd.merge(
    test_transaction,
    test_identity,
    on="TransactionID",
    how="left"
)

print("test_transaction shape:", test_transaction.shape)
print("test_identity shape:", test_identity.shape)
print("merged test shape:", test_df.shape)

test_transaction shape: (506691, 393)
test_identity shape: (141907, 41)
merged test shape: (506691, 433)


In [11]:
test_proba = final_pipeline.predict_proba(test_df)[:, 1]

submission = pd.DataFrame({
    "TransactionID": test_df["TransactionID"],
    "isFraud": test_proba
})

submission.to_csv("submission_xgboost.csv", index=False)

print(submission.shape)
print(submission.head())

(506691, 2)
   TransactionID   isFraud
0        3663549  0.006769
1        3663550  0.036090
2        3663551  0.018866
3        3663552  0.008761
4        3663553  0.027529


In [12]:
print(submission["isFraud"].min())
print(submission["isFraud"].max())
print(submission["isFraud"].describe())
print("Unique prediction values:", submission["isFraud"].nunique())

print("\nSubmission columns:")
print(submission.columns.tolist())

print("\nMissing predictions:")
print(submission["isFraud"].isna().sum())

0.00044407355
0.99985206
count    506691.000000
mean          0.097152
std           0.164164
min           0.000444
25%           0.017453
50%           0.039499
75%           0.093998
max           0.999852
Name: isFraud, dtype: float64
Unique prediction values: 498938

Submission columns:
['TransactionID', 'isFraud']

Missing predictions:
0
